# Decision Tree

## 1. Import Libraries

In [9]:
import pandas as pd
import numpy as np
import os
import pickle

from sklearn.ensemble import StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score)

import warnings
warnings.filterwarnings('ignore')

## Load Data

In [10]:
# 1. LOAD DỮ LIỆU ĐÃ SCALED
data_dir = '../data/'
target_col = 'class'

train_df = pd.read_csv(os.path.join(data_dir, 'train_data.csv'))
valid_df = pd.read_csv(os.path.join(data_dir, 'valid_data.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test_data.csv'))

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_valid = valid_df.drop(columns=[target_col])
y_valid = valid_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

###  Danh sách lưu kết quả để xuất CSV

In [11]:
results_list = []

def evaluate_model(model, name, X_split, y_split, split_name):
    """Dự đoán và trích xuất các chỉ số phân loại theo yêu cầu (Acc, Prec, Rec, F1, AUC)."""
    y_pred = model.predict(X_split)
    y_prob = model.predict_proba(X_split)[:, 1]  # Xác suất class 1 để tính AUC

    acc = accuracy_score(y_split, y_pred)
    prec = precision_score(y_split, y_pred)
    rec = recall_score(y_split, y_pred)
    f1 = f1_score(y_split, y_pred)
    auc = roc_auc_score(y_split, y_prob)

    res = {
        'Algorithm': name,
        'Split': split_name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4),
        'AUC': round(auc, 4)
    }

    results_list.append(res)
    print(f"[{name} | {split_name}] Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")
    return res


### 3. CHẠY BASELINE MODEL (Tham số mặc định)

In [12]:
print("--- V1: Baseline Decision Tree ---")
# Sử dụng cấu hình mặc định để làm mốc so sánh (Baseline)
DecisionTree_baseline_model = DecisionTreeClassifier(random_state=42)
DecisionTree_baseline_model.fit(X_train, y_train)

evaluate_model(DecisionTree_baseline_model, "V1: DecisionTree Baseline", X_valid, y_valid, "Validation")
evaluate_model(DecisionTree_baseline_model, "V1: DecisionTree Baseline", X_test, y_test, "Test")

--- V1: Baseline Decision Tree ---
[V1: DecisionTree Baseline | Validation] Acc: 0.9951 | Prec: 0.9892 | Rec: 1.0000 | F1: 0.9946 | AUC: 0.9955
[V1: DecisionTree Baseline | Test] Acc: 0.9975 | Prec: 0.9946 | Rec: 1.0000 | F1: 0.9973 | AUC: 0.9977


{'Algorithm': 'V1: DecisionTree Baseline',
 'Split': 'Test',
 'Accuracy': 0.9975,
 'Precision': 0.9946,
 'Recall': 1.0,
 'F1-Score': 0.9973,
 'AUC': 0.9977}

### 4. CHẠY TUNING MODEL (Dùng GridSearchCV)

In [13]:
print("--- V2: GridSearchCV Tuning ---")
param_grid = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_depth': [None, 3, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2']
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
DecisionTree_tuned_model = grid_search.best_estimator_

print(f"Best Params found by CV: {grid_search.best_params_}")
evaluate_model(DecisionTree_tuned_model, "V2: DecisionTree Tuned (GridSearch)", X_valid, y_valid, "Validation")
evaluate_model(DecisionTree_tuned_model, "V2: DecisionTree Tuned (GridSearch)", X_test, y_test, "Test")

--- V2: GridSearchCV Tuning ---
Best Params found by CV: {'criterion': 'entropy', 'max_depth': None, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 2}
[V2: DecisionTree Tuned (GridSearch) | Validation] Acc: 0.9803 | Prec: 0.9783 | Rec: 0.9783 | F1: 0.9783 | AUC: 0.9801
[V2: DecisionTree Tuned (GridSearch) | Test] Acc: 0.9802 | Prec: 0.9781 | Rec: 0.9781 | F1: 0.9781 | AUC: 0.9801


{'Algorithm': 'V2: DecisionTree Tuned (GridSearch)',
 'Split': 'Test',
 'Accuracy': 0.9802,
 'Precision': 0.9781,
 'Recall': 0.9781,
 'F1-Score': 0.9781,
 'AUC': 0.9801}

In [14]:
print("--- V3: Ensemble Learning (Stacking) ---")
# Kết hợp Decision Tree và KNN làm Base Models
base_estimators = [
    ('dt', DecisionTreeClassifier(max_depth=5, random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=5))
]

DecisionTree_stacking_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=DecisionTreeClassifier(random_state=42),
    cv=5
)

DecisionTree_stacking_model.fit(X_train, y_train)
evaluate_model(DecisionTree_stacking_model, "V3: DecisionTree Stacking Ensemble", X_valid, y_valid, "Validation")
evaluate_model(DecisionTree_stacking_model, "V3: DecisionTree Stacking Ensemble", X_test, y_test, "Test")

--- V3: Ensemble Learning (Stacking) ---
[V3: DecisionTree Stacking Ensemble | Validation] Acc: 1.0000 | Prec: 1.0000 | Rec: 1.0000 | F1: 1.0000 | AUC: 1.0000
[V3: DecisionTree Stacking Ensemble | Test] Acc: 0.9975 | Prec: 0.9946 | Rec: 1.0000 | F1: 0.9973 | AUC: 0.9955


{'Algorithm': 'V3: DecisionTree Stacking Ensemble',
 'Split': 'Test',
 'Accuracy': 0.9975,
 'Precision': 0.9946,
 'Recall': 1.0,
 'F1-Score': 0.9973,
 'AUC': 0.9955}

### (5) Lưu kết quả

In [15]:
# CẤU HÌNH ĐƯỜNG DẪN LƯU
save_path = '../experiment/Decision Tree/'
os.makedirs(save_path, exist_ok=True)

# 1. TỔNG HỢP & LƯU 1 FILE CSV DUY NHẤT
df_results = pd.DataFrame(results_list)

csv_output = os.path.join(save_path, 'Decision_Tree_all_results.csv')
df_results.to_csv(csv_output, index=False)

# 2. LƯU MODELS (.pkl)
with open(os.path.join(save_path, 'Decision_Tree_baseline.pkl'), 'wb') as f:
    pickle.dump(DecisionTree_baseline_model, f)

with open(os.path.join(save_path, 'Decision_Tree_tuned.pkl'), 'wb') as f:
    pickle.dump(DecisionTree_tuned_model, f)

with open(os.path.join(save_path, 'Decision_Tree_stacking.pkl'), 'wb') as f:
    pickle.dump(DecisionTree_stacking_model, f)

print("-" * 40)
print(f"✅ Đã lưu tất cả kết quả vào: {csv_output}")
print(f"✅ Đã lưu models tại: {save_path}")
print("-" * 40)

display(df_results)

----------------------------------------
✅ Đã lưu tất cả kết quả vào: ../experiment/Decision Tree/Decision_Tree_all_results.csv
✅ Đã lưu models tại: ../experiment/Decision Tree/
----------------------------------------


,Algorithm,Split,Accuracy,Precision,Recall,F1-Score,AUC
0,V1: DecisionTree Baseline,Validation,0.9951,0.9892,1.0000,0.9946,0.9955
1,V1: DecisionTree Baseline,Test,0.9975,0.9946,1.0000,0.9973,0.9977
2,V2: DecisionTree Tuned (GridSearch),Validation,0.9803,0.9783,0.9783,0.9783,0.9801
3,V2: DecisionTree Tuned (GridSearch),Test,0.9802,0.9781,0.9781,0.9781,0.9801
4,V3: DecisionTree Stacking Ensemble,Validation,1.0000,1.0000,1.0000,1.0000,1.0000
5,V3: DecisionTree Stacking Ensemble,Test,0.9975,0.9946,1.0000,0.9973,0.9955


In [16]:
!jupyter nbconvert --to html Decision_Tree_model.ipynb

[NbConvertApp] Converting notebook Decision_Tree_model.ipynb to html
[NbConvertApp] Writing 306721 bytes to Decision_Tree_model.html
